# 04. MAF でシングルエージェントを構築する (MCP ツールあり)

## コードの解説

`MCPStreamableHTTPTool` を `tools=[]` に渡すだけで、MCP サーバーが提供するすべてのツールをエージェントが使えるようになります。
エージェントは会話の文脈に応じて **自動的に適切なツールを選択・実行** します。

### 前提条件

このノートブックを実行する前に、MCP サーバーを起動してください：

```bash
python mcp/mock_server.py
```

### ツールなしとの違い

| 項目 | ツールなし (ws_03) | MCP ツールあり (ws_04) |
|-----|----------------|--------------------|
| 情報源 | LLM 学習データのみ | LLM + MCP サーバーのリアルタイムデータ |
| 精度 | 汎用的な知識 | 最新・正確なデータ |
| tools 引数 | 省略 | `tools=[mcp_tool]` |

### Agent とツールの関係

```
Agent
  ├─ client: OpenAIChatCompletionClient  ← LLM（推論・応答生成）
  └─ tools: [MCPStreamableHTTPTool]      ← 外部データアクセス
              │
              └─→ MCP サーバー (mock_server.py)
                    ├─ 製品情報検索
                    ├─ 注文管理
                    └─ 在庫確認 など
```

In [ ]:
import os
from dotenv import load_dotenv
from agent_framework import Agent, MCPStreamableHTTPTool
from agent_framework.openai import OpenAIChatCompletionClient

load_dotenv()

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY  = os.getenv("AZURE_OPENAI_API_KEY")
MODEL_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
MCP_SERVER_URI        = os.getenv("MCP_SERVER_URI", "http://localhost:8000/mcp")

USE_KEY_AUTH = os.getenv("USE_KEY_AUTH", "False").lower() in ("true", "1", "t")
if USE_KEY_AUTH:
    auth_kwargs = dict(api_key=AZURE_OPENAI_API_KEY)
else:
    from azure.identity.aio import DefaultAzureCredential
    os.environ.pop("AZURE_OPENAI_API_KEY", None)
    auth_kwargs = dict(credential=DefaultAzureCredential())


# ---------------------------------------------------------------
# MCPStreamableHTTPTool の作成
# MCP サーバーへの接続情報を定義する
# ---------------------------------------------------------------
mcp_tool = MCPStreamableHTTPTool(
    name="game_shop_tools",
    url=MCP_SERVER_URI,
    description="Xbox ゲームショップの製品・注文・在庫情報を取得するツール群です。",
    timeout=120,
)


# ---------------------------------------------------------------
# MCP ツール付きエージェントの作成
#
# ポイント: tools=[mcp_tool] で MCP ツールを渡す
#   → エージェントはユーザーの質問に応じて MCP サーバーのツールを自動選択・実行する
# ---------------------------------------------------------------
agent = Agent(
    client=OpenAIChatCompletionClient(
        **auth_kwargs,
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
        model=MODEL_DEPLOYMENT_NAME,
    ),
    name="GameShopAssistant",
    instructions=(
        "あなたは Xbox ゲームショップのアシスタントです。"
        "MCP ツールを使って製品情報や注文情報を検索し、お客様の質問に答えてください。"
        "情報が見つからない場合は正直にその旨をお伝えください。"
    ),
    tools=[mcp_tool],
)

# エージェントの実行
response = await agent.run("Xbox Series X の価格はいくらですか？")
print(response)